# Ferramentas

Um modelo de linguagem produz tokens, e nada além disso. Calcular com exatidão ou gravar um arquivo acontece fora dele, em código Python que alguém escreveu.

Ferramenta é o arranjo que liga as duas coisas, em quatro passos: declarar as funções no prompt, pedir a execução em um formato conhecido, executar a função e devolver o resultado no contexto. Este notebook percorre os quatro passos e termina no laço que os repete.

In [ ]:
# No Google Colab, descomente e rode uma vez (Ambiente de execução > GPU).
# !pip install -q "agentkit @ git+https://github.com/silvaan/agentic-ai"

import inspect
import json
import urllib.request
from pathlib import Path
from typing import Callable, get_type_hints

import torch

from agentkit import LLM

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
llm = LLM(MODEL_NAME, device=device, temperature=0.0, max_tokens=200)
print(llm.model)

## Limites da geração de texto

As duas células seguintes pedem coisas que parecem simples. Antes de rodar, tente prever o resultado de cada uma.

In [ ]:
print(llm.invoke([{"role": "user", "content": "Quanto é 4871 vezes 3926? Responda apenas com o número."}]))
print(f"resposta correta: {4871 * 3926}")

A multiplicação exata exige um algoritmo com transporte entre casas. O modelo prevê tokens plausíveis para um número dessa forma, e devolveu oito dígitos com o valor errado.

In [ ]:
print(llm.invoke([
    {"role": "user", "content": "Salve um resumo de uma linha sobre a linguagem Python em um arquivo chamado resumo.txt."},
], max_tokens=100))
print(f"arquivo existe: {Path('resumo.txt').exists()}")

O modelo devolveu o código Python que resolveria o pedido, e nenhum arquivo apareceu, porque nada foi executado. Os dois pedidos têm a mesma causa: a saída é texto, e texto não multiplica nem grava em disco.

## Protocolo textual de chamada

O formato de chamada é uma convenção de texto escrita no prompt. A convenção precisa de três coisas: quais funções existem, como pedir uma delas e o que fazer quando o resultado chegar.

In [ ]:
def calculate(expression: str) -> str:
    """Avalia uma expressão aritmética, como 12 * (3 + 4)."""
    return str(eval(expression, {"__builtins__": {}}, {}))


print(calculate("4871 * 3926"))

A avaliação sem builtins bloqueia `open`, `__import__` e o resto da biblioteca padrão a partir da string, e é a proteção mínima para executar algo que veio do modelo. A função é declarada ao modelo pela mensagem de sistema abaixo.

In [ ]:
SYSTEM = """Você é um assistente com acesso a ferramentas.

# Ferramentas
calculate(expression: str): avalia uma expressão aritmética e devolve o resultado.

# Regras
Você não faz contas de cabeça: toda conta é feita pela ferramenta.
Para chamar a ferramenta, responda apenas com uma linha, sem nenhum outro texto:
CHAMAR: {"name": "calculate", "arguments": {"expression": "..."}}"""

question = "Quanto é 4871 vezes 3926?"
messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
answer = llm.invoke(messages)
print(answer)

O modelo respondeu com a linha combinada, no lugar do número inventado da primeira seção. Existe uma intenção de chamada escrita em texto, e nada foi executado ainda.

### Execução e observação

Executar é ler a linha, virar dicionário e chamar a função. O resultado entra na conversa como uma mensagem nova, com o papel `tool`, depois do turno do assistente que pediu a chamada.

In [ ]:
def parse_call(text: str) -> dict | None:
    """Lê a linha CHAMAR e devolve o dicionário da chamada, ou None se não houver."""
    for line in text.splitlines():
        if line.strip().startswith("CHAMAR:"):
            return json.loads(line.split("CHAMAR:", 1)[1])
    return None


call = parse_call(answer)
result = calculate(**call["arguments"])
print(call, result)

In [ ]:
messages += [
    {"role": "assistant", "content": answer},
    {"role": "tool", "name": call["name"], "content": result},
]

rendered = llm.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(rendered[rendered.index("<|im_start|>assistant"):])

O papel `tool` existe na lista de mensagens e não no prompt: o template o converte em um turno de usuário com o conteúdo dentro de `<tool_response>`. A lista de dicionários é a estrutura de programação, e o texto renderizado é o que o modelo lê.

In [ ]:
print(llm.invoke(messages, max_tokens=80))

O ciclo fechou e a resposta final traz o número que veio da execução. Tudo que existe aqui é prompt, string e uma chamada de função.

### Exercício 1

Acrescente uma segunda função à lista de ferramentas da mensagem de sistema, como uma que conte as palavras de um texto, e faça um pedido que exija a nova função. Responda se o modelo escolheu a função certa e se manteve o formato da linha `CHAMAR:`.

In [ ]:
# Seu código aqui

## Ferramenta como contrato

A ferramenta é um contrato entre o programa e o modelo: o esquema declara nome, parâmetros e tipos, e a docstring diz para que ela serve. Escrever isso à mão no prompt duplica o que a função já traz na assinatura, e o decorador abaixo lê as duas coisas por introspecção.

In [ ]:
TYPE_NAMES = {str: "string", int: "integer", float: "number", bool: "boolean"}


def tool(fn: Callable) -> Callable:
    """Anexa fn.tool_schema por introspecção e devolve a própria função."""
    hints = get_type_hints(fn)
    names = list(inspect.signature(fn).parameters)
    fn.tool_schema = {
        "name": fn.__name__,
        "description": inspect.getdoc(fn).splitlines()[0],
        "parameters": {name: {"type": TYPE_NAMES[hints[name]]} for name in names},
        "required": names,
    }
    return fn

In [ ]:
tool(calculate)
print(json.dumps(calculate.tool_schema, indent=2, ensure_ascii=False))

O decorador devolve a própria função, que continua chamável sem modelo nenhum, e escrever `@tool` acima da definição é essa mesma chamada. A tabela de tipos tem quatro entradas de propósito: um tipo não previsto levanta `KeyError` na linha do decorador, e não em produção. A primeira linha da docstring é copiada para dentro do prompt, então ela acompanha a língua da conversa.

### Marcadores escritos pelo template

A convenção da seção anterior foi escrita por nós. Modelos ajustados para ferramentas trazem uma convenção própria, aprendida no treino, e o template de conversa a escreve a partir dos esquemas passados em `tools`.

In [ ]:
prompt = llm.tokenizer.apply_chat_template(
    [{"role": "user", "content": question}],
    tools=[calculate.tool_schema], tokenize=False, add_generation_prompt=True,
)
print(prompt)

O template escreveu o esquema em um bloco `<tools>` dentro da mensagem de sistema e mandou responder com um objeto JSON entre `<tool_call>` e `</tool_call>`. Essas instruções vêm em inglês porque foram escritas no ajuste do modelo, e a descrição da ferramenta entrou como está na docstring.

In [ ]:
llm_with_tools = llm.bind_tools([calculate])
message = llm_with_tools.invoke([{"role": "user", "content": question}])
message

Com ferramentas ligadas, `invoke` devolve a mensagem do assistente em vez do texto, com a chamada normalizada em uma lista de dicionários com `name` e `arguments`. Quando o modelo responde sem pedir ferramenta, a mensagem vem só com `content`, e é a ausência de `tool_calls` que encerra o laço.

### Despacho e execução

Com a chamada normalizada, executar é uma tabela de nome para função. A saída é sempre texto, porque ela vai entrar na conversa como observação.

In [ ]:
def run_tool(call: dict, tools: dict) -> str:
    """Executa a ferramenta pedida e devolve o resultado como texto."""
    fn = tools.get(call["name"])
    if fn is None:
        return f"ferramenta desconhecida: {call['name']}"
    try:
        return str(fn(**call["arguments"]))
    except Exception as error:
        return f"{type(error).__name__}: {error}"

In [ ]:
available = {"calculate": calculate}

print(run_tool({"name": "calculate", "arguments": {"expression": "37 * 42"}}, available))
print(run_tool({"name": "calculate", "arguments": {"expression": "1 / 0"}}, available))
print(run_tool({"name": "soma", "arguments": {"a": 1, "b": 2}}, available))

O primeiro é o caso normal, o segundo levanta exceção e o terceiro traz um nome fora da tabela. Os dois últimos viram texto descrevendo a falha, e nenhum interrompe a execução, porque quem decide o que fazer com o erro é o modelo no passo seguinte.

### Exercício 2

Escreva uma ferramenta sua com `@tool`, com docstring de uma linha e parâmetros anotados, e ligue-a ao modelo junto de `calculate`. Faça um pedido que exija a sua ferramenta e responda qual delas o modelo pediu e com quais argumentos.

In [ ]:
# Seu código aqui

## Laço de agente

Até aqui a execução foi feita por nós, um passo de cada vez. Quando a resposta ao usuário depende do que a ferramenta devolveu, alguém precisa repetir o ciclo até o modelo parar de pedir ferramenta.

In [ ]:
class Agent:
    """Modelo com ferramentas ligadas e o laço que alterna chamada e execução."""

    def __init__(self, llm: LLM, tools: list[Callable], max_steps: int = 5) -> None:
        self.llm = llm.bind_tools(tools)
        self.tools = {fn.tool_schema["name"]: fn for fn in tools}
        self.max_steps = max_steps

    def run(self, question: str) -> list[dict]:
        """Repete o ciclo até o modelo responder sem pedir ferramenta."""
        messages = [{"role": "user", "content": question}]
        for _ in range(self.max_steps):
            message = self.llm.invoke(messages)
            messages.append(message)
            if "tool_calls" not in message:
                break
            for call in message["tool_calls"]:
                result = run_tool(call, self.tools)
                messages.append({"role": "tool", "name": call["name"], "content": result})
        return messages

O laço não menciona template, marcador nem parse: pede a mensagem, executa o que ela pedir, devolve a observação e repete. O `max_steps` é a proteção contra laço infinito, e sem ele um modelo que insiste em chamar a mesma ferramenta roda para sempre. Essa mesma classe existe no `agentkit`, em `agent.py`.

In [ ]:
for message in Agent(llm, [calculate]).run(question):
    print(message["role"], ":", message.get("content") or message["tool_calls"])

Quatro mensagens: a pergunta, o turno que pede a chamada, a observação e a resposta final. O laço parou porque a última mensagem veio sem `tool_calls`.

### Exercício 3

Escreva o encadeamento à mão, sem laço nenhum: execute `calculate` em uma conta de sua escolha e passe o resultado para `write_file`. Depois entregue a mesma tarefa em uma frase só a um `Agent` com as duas ferramentas e responda o que ele fez com o argumento que dependia do resultado da primeira chamada.

In [ ]:
# Seu código aqui

## Arquivos e serviços externos

O `calculate` recebe uma expressão e devolve um número, sem tocar em nada fora do processo. As duas ferramentas seguintes fazem o que a conta não faz: uma escreve no disco e a outra busca um dado que não está nos pesos e muda a cada hora.

### Escrita e leitura de arquivos

Ferramenta com efeito colateral precisa de fronteira. O `resolve` recusa nomes que tentem sair do diretório de trabalho, antes de qualquer acesso, e as duas ferramentas passam por ele.

In [ ]:
WORKSPACE = Path("workspace")


def resolve(name: str) -> Path:
    """Recusa nomes que apontem fora do diretório de trabalho."""
    WORKSPACE.mkdir(exist_ok=True)
    target = (WORKSPACE / name).resolve()
    if not target.is_relative_to(WORKSPACE.resolve()):
        raise ValueError("caminho fora do diretório de trabalho")
    return target


@tool
def write_file(name: str, content: str) -> str:
    """Escreve um texto em um arquivo do diretório de trabalho."""
    resolve(name).write_text(content, encoding="utf-8")
    return f"{len(content)} caracteres escritos em {name}"


@tool
def read_file(name: str) -> str:
    """Lê um arquivo do diretório de trabalho e devolve o conteúdo."""
    return resolve(name).read_text(encoding="utf-8")

Os dois pedidos abaixo são o da abertura do notebook e a leitura do que ele gravou. O texto do resumo é escrito pelo próprio modelo, e o arquivo aparece no disco.

In [ ]:
files_agent = Agent(llm, [write_file, read_file])

for request in ["Salve um resumo de uma linha sobre a linguagem Python no arquivo resumo.txt.",
                "O que está escrito no arquivo resumo.txt?"]:
    for message in files_agent.run(request):
        print(message["role"], ":", message.get("content") or message["tool_calls"])
    print()

O arquivo apareceu no disco, com o texto que o próprio modelo escreveu, e é a segunda falha da abertura resolvida. O nome saiu como `resumos`, sem a extensão que o pedido trazia, e a leitura seguinte falhou por isso: a exceção voltou como observação e o agente relatou a falha em vez de inventar o conteúdo.

### Consulta a uma API

A ferramenta abaixo consulta um serviço público de previsão do tempo, sem chave de acesso. Ela traz para o contexto um dado que o modelo não teria de outro jeito.

In [ ]:
@tool
def get_temperature(latitude: float, longitude: float) -> str:
    """Consulta a temperatura atual, em graus Celsius, de uma latitude e uma longitude."""
    url = (
        "https://api.open-meteo.com/v1/forecast"
        f"?latitude={latitude}&longitude={longitude}&current=temperature_2m"
    )
    with urllib.request.urlopen(url, timeout=10) as response:
        return str(json.load(response)["current"]["temperature_2m"])


for message in Agent(llm, [get_temperature]).run("Qual é a temperatura atual em Natal? As coordenadas são -5.79, -35.21."):
    print(message["role"], ":", message.get("content") or message["tool_calls"])

A observação é o dado do serviço, e o modelo leu as coordenadas do texto do pedido. Uma ferramenta assim traz três problemas que as anteriores não têm: latência variável, falha fora do controle do programa e resposta que muda entre execuções, o que impede comparar duas rodadas por igualdade de texto.

### Exercício 4

`DATABASE` é um dicionário no lugar de um banco. Escreva uma ferramenta que consulta o cadastro pelo e-mail e outra que acrescenta créditos, monte um `Agent` com as duas e rode os quatro pedidos.

Responda qual ferramenta o modelo pediu em cada um, com que argumentos, e o que ele fez com o e-mail que não existe.

In [ ]:
DATABASE = {
    "ana@acme.com": {"nome": "Ana Lima", "plano": "atlas", "creditos": 120},
    "bruno@acme.com": {"nome": "Bruno Sá", "plano": "orbit", "creditos": 0},
    "dora@acme.com": {"nome": "Dora Reis", "plano": "atlas", "creditos": 45},
}

DATABASE_REQUESTS = [
    "Quantos créditos a Ana tem? O e-mail dela é ana@acme.com.",
    "Adicione 50 créditos para bruno@acme.com.",
    "Qual é o plano de dora@acme.com?",
    "Consulte o cadastro de joao@acme.com.",
]

# Seu código aqui